In [ ]:
import pandas as pd
import numpy as np
import json
import math

#def calculate_bond_rating(all,yield_percent, risk, duration_months, price,
#                   k1=0.05, k2=0.02, k3=0.1):
def calculate_bond_rating(arr, k1=0.05, k2=0.02, k3=0.1):
    """
    Расчёт рейтинга облигации с нелинейным штрафом за риск и дюрацию (в месяцах).


    Параметры:
    - yield_percent (float): доходность в процентах (например, 8.25)
    - risk (int/float): уровень риска (например, 1)
    - duration_months (int): дюрация в месяцах (например, 26)
    - price (float): цена облигации в рублях (например, 985.50)
    - k1 (float): коэффициент штрафа за риск (по умолчанию 0.05)
    - k2 (float): коэффициент штрафа за дюрацию (по умолчанию 0.02)
    - k3 (float): вес дисконта к номиналу (по умолчанию 0.1)

    Возвращает:
    - float: рейтинг облигации (чем выше, тем лучше)
    """
    yield_percent, risk, duration_months, price = arr[:,0], arr[:,1], arr[:,2], arr[:,3]
    # Нормализация доходности (перевод процентов в доли)
    normalized_yield = yield_percent / 100

    # Штраф за риск: квадратичная зависимость
    risk_penalty = k1 * (risk ** 2)

    # Штраф за дюрацию: квадратичная зависимость, нормализация на 36 месяцев (3 года)
    duration_penalty = k2 * ((duration_months / 36) ** 2)

    # Дисконт к номиналу (номинал = 1000 руб.)
    discount = (1000 - price) / 1000

    # Расчёт итогового рейтинга
    rating = (
        normalized_yield *
        (1 - risk_penalty) *
        (1 - duration_penalty) +
        k3 * discount
    )
    return abs(rating)

def json_to_dataframe(json_path=None, json_data=None):
    """
    Преобразует JSON в DataFrame.
    Можно передать путь к файлу или уже загруженные данные.
    """
    try:
        if json_path:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        else:
            data = json_data

        # Если данные — список объектов
        if isinstance(data, list):
            df = pd.DataFrame(data)
        # Если данные содержат вложенную структуру
        elif isinstance(data, dict) and 'bonds' in data:
            df = pd.json_normalize(data['bonds'])
        else:
            # Универсальный вариант
            df = pd.json_normalize(data)

        return df

    except Exception as e:
        print(f"Ошибка при преобразовании JSON: {e}")
        return pd.DataFrame()

# Использование функции, установка названием строк "Название" облигации
df = json_to_dataframe(json_path='data.json')
#df.set_index('name',inplace=True)

# Математический подсчёт рейтинга каждой облигации
df['price_sale']=(df['price']*(df['yield_percent'])/100)/df['duration']

# Пример использования с вашими данными
df['rating'] = calculate_bond_rating(df.loc[:,'price':'duration'].values)

# Подготовка к выводу, вывод конечных данных
#df.reset_index(inplace=True)
#print(df)
df.to_json('export.json',orient='records',index=False)


         ticker                     name    price  yield_percent  risk  \
0  SU26207RMFS3                ОФЗ 26207   985.50           8.25     1   
1  RU000A1012Q5   Облигация Газпром 2025  1020.00           7.80     2   
2  SU26210RMFS1                ОФЗ 26210   995.75           8.10     1   
3  RU000A0JWTN3  Облигация Сбербанк 2026  1015.20           8.40     1   
4  RU000A1023R7       Облигация ВТБ 2027   970.80           9.15     2   

   duration  price_sale     rating  
0        26    3.127067  23.585031  
1        37    2.150270  20.730814  
2        54    1.493625  22.613128  
3        23    3.707687  25.566160  
4        62    1.432713  30.835192  
